In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)

/home/hanxd/Repositories/substrateSimilarity/ESPCalculate/train_model


In [ ]:
df_test = pd.read_pickle(CURRENT_DIR + "/../data/df_test_with_ESM1b_ts.pkl")
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(CURRENT_DIR + "/../data/df_test_with_ESM1b_ts.pkl")
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [ ]:
ugt = pd.read_pickle(CURRENT_DIR + "/../data/trainUGT.pkl")

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
train_X, train_y = create_input_and_output_data(df=df_train)
test_X, test_y = create_input_and_output_data(df=df_test)
test_new_X, test_new_y = create_input_and_output_data(df=ugt)

from sklearn.model_selection import train_test_split

train_X = np.concatenate([train_X, test_new_X])
train_y = np.concatenate([train_y, test_new_y])

In [ ]:
param = {
    "learning_rate": 0.60553117247348733,
    "max_delta_step": 1.7726044219753656,
    "max_depth": 10,
    "min_child_weight": 1.3845040588450772,
    "num_rounds": 342.68325188584106,
    "reg_alpha": 0.531395259755843,
    "reg_lambda": 3.744980563764689,
    "weight": 0.26187490421514203,
}

num_round = param["num_rounds"]
# param["tree_method"] = "gpu_hist"
# param["sampling_method"] = "gradient_based"
param["objective"] = "binary:logistic"
param["eval_metric"] = ["error", "logloss"]

weights1 = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in df_train["Binding"]]
)
weights2 = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in test_new_y]
)

weights = np.concatenate([weights1, weights2])


del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(
    np.array(train_X),
    weight=weights,
    label=np.array(train_y),
    feature_names=feature_names,
)
dtest = xgb.DMatrix(
    np.array(test_X), label=np.array(test_y), feature_names=feature_names
)

evallist = [(dtrain, "train")]

bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc))

[0]	train-error:0.28277	train-logloss:0.59476
[10]	train-error:0.07475	train-logloss:0.30925
[20]	train-error:0.03987	train-logloss:0.21823
[30]	train-error:0.02252	train-logloss:0.16160
[40]	train-error:0.01401	train-logloss:0.12647
[50]	train-error:0.00953	train-logloss:0.10413
[60]	train-error:0.00753	train-logloss:0.08851
[70]	train-error:0.00561	train-logloss:0.07612
[80]	train-error:0.00471	train-logloss:0.06779
[90]	train-error:0.00420	train-logloss:0.06097
[100]	train-error:0.00380	train-logloss:0.05554
[110]	train-error:0.00357	train-logloss:0.05132
[120]	train-error:0.00333	train-logloss:0.04736
[130]	train-error:0.00330	train-logloss:0.04455
[140]	train-error:0.00302	train-logloss:0.04189
[150]	train-error:0.00290	train-logloss:0.03989
[160]	train-error:0.00275	train-logloss:0.03801
[170]	train-error:0.00275	train-logloss:0.03650
[180]	train-error:0.00267	train-logloss:0.03504
[190]	train-error:0.00259	train-logloss:0.03370
[200]	train-error:0.00255	train-logloss:0.03259
[21

In [ ]:
with open(os.path.join(CURRENT_DIR, "../model/ugtnormalmodel.pkl"), "wb") as file:
    pickle.dump(bst, file)